# Splitting Raw Datasets into 2023–2024 and 2025 (Jan–Nov) Subsets & Generating BW Holidays

This notebook splits the 3 raw Excel datasets in `Dataset/raw/` into 6 subsets covering **2023–2024** and **2025 (excluding December due to missing values in Day-Ahead Prices)** while preserving each file's original metadata preamble and generates the Baden-Württemberg public holiday dataset (`df_holiday.xlsx`):

1. **Actual Consumption** (`Actual_consumption_201701010000_202601010000_Quarterhour.xlsx`)
   - `Actual_consumption_2023_2024_Quarterhour.xlsx`
   - `Actual_consumption_2025_Quarterhour.xlsx`
2. **Day-Ahead Prices** (`Day-ahead_prices_201701010000_202601010000_Quarterhour.xlsx`)
   - `Day-ahead_prices_2023_2024_Quarterhour.xlsx`
   - `Day-ahead_prices_2025_Quarterhour.xlsx`
3. **Open-Meteo Weather** (`open-meteo-48.83N9.15E293m.xlsx`)
   - `open-meteo-48.83N9.15E293m_2023_2024.xlsx`
   - `open-meteo-48.83N9.15E293m_2025.xlsx`
4. **Baden-Württemberg Public Holidays** (`df_holiday.xlsx`)

In [2]:
import locale
from pathlib import Path
import holidays
import pandas as pd

# Ensure English month names ('Jan', 'Mar', 'May', 'Oct', 'Nov', 'Dec') and 'AM'/'PM' parse across OS locales
locale.setlocale(locale.LC_TIME, "C")

# Resolve Dataset/raw directory whether run from Dataset/raw or repository root
cwd = Path.cwd().resolve()
RAW_DIR = cwd if cwd.name == "raw" else cwd / "Dataset" / "raw"
HOLIDAY_FILE = RAW_DIR / "df_holiday.xlsx"

DATASET_CONFIGS = [
    {
        "name": "Actual Consumption",
        "input": RAW_DIR / "Actual_consumption_201701010000_202601010000_Quarterhour.xlsx",
        "skiprows": 9,
        "ts_col": "Start date",
        "ts_format": "%b %d, %Y %I:%M %p",
        "out_2023_2024": RAW_DIR / "Actual_consumption_2023_2024_Quarterhour.xlsx",
        "out_2025": RAW_DIR / "Actual_consumption_2025_Quarterhour.xlsx",
    },
    {
        "name": "Day-Ahead Prices",
        "input": RAW_DIR / "Day-ahead_prices_201701010000_202601010000_Quarterhour.xlsx",
        "skiprows": 9,
        "ts_col": "Start date",
        "ts_format": "%b %d, %Y %I:%M %p",
        "out_2023_2024": RAW_DIR / "Day-ahead_prices_2023_2024_Quarterhour.xlsx",
        "out_2025": RAW_DIR / "Day-ahead_prices_2025_Quarterhour.xlsx",
    },
    {
        "name": "Open-Meteo Weather",
        "input": RAW_DIR / "open-meteo-48.83N9.15E293m.xlsx",
        "skiprows": 3,
        "ts_col": "time",
        "ts_format": None,
        "out_2023_2024": RAW_DIR / "open-meteo-48.83N9.15E293m_2023_2024.xlsx",
        "out_2025": RAW_DIR / "open-meteo-48.83N9.15E293m_2025.xlsx",
    },
]

In [3]:
def save_with_preamble(preamble_df: pd.DataFrame, header_row: pd.DataFrame, data_df: pd.DataFrame, out_path: Path) -> None:
    """Save split data rows together with the original Excel metadata preamble and header row."""
    combined = pd.concat([preamble_df, header_row, data_df], ignore_index=True)
    combined.to_excel(out_path, index=False, header=False)


summary_records = []

for cfg in DATASET_CONFIGS:
    raw_all = pd.read_excel(cfg["input"], header=None)
    skip = cfg["skiprows"]

    preamble = raw_all.iloc[:skip]
    header_row = raw_all.iloc[skip : skip + 1]
    data = raw_all.iloc[skip + 1 :].copy()

    col_names = list(header_row.iloc[0])
    ts_idx = col_names.index(cfg["ts_col"])
    ts_series = data[ts_idx]

    if pd.api.types.is_datetime64_any_dtype(ts_series):
        parsed_ts = ts_series
    elif cfg["ts_format"]:
        parsed_ts = pd.to_datetime(ts_series, format=cfg["ts_format"], errors="coerce")
    else:
        parsed_ts = pd.to_datetime(ts_series, errors="coerce")

    years = parsed_ts.dt.year
    months = parsed_ts.dt.month
    df_23_24 = data.loc[years.isin([2023, 2024])]
    df_25 = data.loc[(years == 2025) & (months < 12)]

    save_with_preamble(preamble, header_row, df_23_24, cfg["out_2023_2024"])
    save_with_preamble(preamble, header_row, df_25, cfg["out_2025"])

    print(cfg["input"].name)
    print(f"  2023-2024: {len(df_23_24):,} rows | {df_23_24[ts_idx].iloc[0]} -> {df_23_24[ts_idx].iloc[-1]} -> {cfg['out_2023_2024'].name}")
    print(f"  2025     : {len(df_25):,} rows | {df_25[ts_idx].iloc[0]} -> {df_25[ts_idx].iloc[-1]} -> {cfg['out_2025'].name}")

    summary_records.extend([
        {
            "dataset": cfg["name"],
            "period": "2023-2024",
            "file": cfg["out_2023_2024"].name,
            "rows": len(df_23_24),
            "start": str(df_23_24[ts_idx].iloc[0]),
            "end": str(df_23_24[ts_idx].iloc[-1]),
        },
        {
            "dataset": cfg["name"],
            "period": "2025 (Jan-Nov)",
            "file": cfg["out_2025"].name,
            "rows": len(df_25),
            "start": str(df_25[ts_idx].iloc[0]),
            "end": str(df_25[ts_idx].iloc[-1]),
        },
    ])

c:\Users\Win11\Documents\GitHub\Energy-Forecasting\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Actual_consumption_201701010000_202601010000_Quarterhour.xlsx
  2023-2024: 70,176 rows | Jan 1, 2023 12:00 AM -> Dec 31, 2024 11:45 PM -> Actual_consumption_2023_2024_Quarterhour.xlsx
  2025     : 32,064 rows | Jan 1, 2025 12:00 AM -> Nov 30, 2025 11:45 PM -> Actual_consumption_2025_Quarterhour.xlsx


c:\Users\Win11\Documents\GitHub\Energy-Forecasting\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Day-ahead_prices_201701010000_202601010000_Quarterhour.xlsx
  2023-2024: 70,176 rows | Jan 1, 2023 12:00 AM -> Dec 31, 2024 11:45 PM -> Day-ahead_prices_2023_2024_Quarterhour.xlsx
  2025     : 32,064 rows | Jan 1, 2025 12:00 AM -> Nov 30, 2025 11:45 PM -> Day-ahead_prices_2025_Quarterhour.xlsx
open-meteo-48.83N9.15E293m.xlsx
  2023-2024: 70,176 rows | 2023-01-01 00:00:00 -> 2024-12-31 23:45:00 -> open-meteo-48.83N9.15E293m_2023_2024.xlsx
  2025     : 32,064 rows | 2025-01-01 00:00:00 -> 2025-11-30 23:45:00 -> open-meteo-48.83N9.15E293m_2025.xlsx


In [4]:
pd.DataFrame(summary_records)

,dataset,period,file,rows,start,end
0,Actual Consumption,2023-2024,Actual_consumption_2023_2024_Quarterhour.xlsx,70176,"Jan 1, 2023 12:00 AM","Dec 31, 2024 11:45 PM"
1,Actual Consumption,2025 (Jan-Nov),Actual_consumption_2025_Quarterhour.xlsx,32064,"Jan 1, 2025 12:00 AM","Nov 30, 2025 11:45 PM"
2,Day-Ahead Prices,2023-2024,Day-ahead_prices_2023_2024_Quarterhour.xlsx,70176,"Jan 1, 2023 12:00 AM","Dec 31, 2024 11:45 PM"
3,Day-Ahead Prices,2025 (Jan-Nov),Day-ahead_prices_2025_Quarterhour.xlsx,32064,"Jan 1, 2025 12:00 AM","Nov 30, 2025 11:45 PM"
4,Open-Meteo Weather,2023-2024,open-meteo-48.83N9.15E293m_2023_2024.xlsx,70176,2023-01-01 00:00:00,2024-12-31 23:45:00
5,Open-Meteo Weather,2025 (Jan-Nov),open-meteo-48.83N9.15E293m_2025.xlsx,32064,2025-01-01 00:00:00,2025-11-30 23:45:00


## Generate & Save Baden-Württemberg (BW) Holiday Dataset (`df_holiday.xlsx`)

In [5]:
bw_holidays = holidays.Germany(years=range(2023, 2025), subdiv="BW")
df_holidays = (
    pd.DataFrame(list(bw_holidays.items()), columns=["Datum", "Feiertag"])
    .sort_values("Datum")
    .reset_index(drop=True)
)
df_holidays["Datum"] = pd.to_datetime(df_holidays["Datum"])
df_holidays.to_excel(HOLIDAY_FILE, index=False)

print(f"Saved {len(df_holidays)} BW holidays to: {HOLIDAY_FILE.name}")
df_holidays[df_holidays["Datum"].dt.year.isin([2023, 2024, 2025])].reset_index(drop=True)

Saved 24 BW holidays to: df_holiday.xlsx


,Datum,Feiertag
0,2023-01-01,Neujahr
1,2023-01-06,Heilige Drei Könige
2,2023-04-07,Karfreitag
3,2023-04-10,Ostermontag
4,2023-05-01,Erster Mai
5,2023-05-18,Christi Himmelfahrt
6,2023-05-29,Pfingstmontag
7,2023-06-08,Fronleichnam
8,2023-10-03,Tag der Deutschen Einheit
9,2023-11-01,Allerheiligen
